In [11]:
!pip install -q unsloth
from unsloth import FastLanguageModel

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

print(type(model))

Unsloth 2026.9.11 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<class 'peft.peft_model.PeftModelForCausalLM'>


In [13]:
from datasets import load_dataset

dataset = load_dataset(
    "databricks/databricks-dolly-15k",
    split="train[:100]"
)

print(dataset)
print(f"dataset sample: {dataset[0]}")


Dataset({
    features: ['instruction', 'context', 'response', 'category'],
    num_rows: 100
})
dataset sample: {'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [14]:
def formatting_func(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

dataset = dataset.map(
    lambda example: {
        "text": formatting_func(example)
    }
)

print(dataset[0]["text"])

formatting_func(dataset[0])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
When did Virgin Australia start operating?<|im_end|>
<|im_start|>assistant
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>



'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhen did Virgin Australia start operating?<|im_end|>\n<|im_start|>assistant\nVirgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.<|im_end|>\n'

In [15]:
def tokenize_example(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=2048,
    )

In [16]:
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        output_dir="outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [17]:
print(type(model))

print("\nPEFT config:")
print(getattr(model, "peft_config", "NO PEFT CONFIG"))

print("\nTrainable parameters:")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable:,}")
print(f"Total:     {total:,}")
print(f"Percent:   {100 * trainable / total:.2f}%")

<class 'peft.peft_model.PeftModelForCausalLM'>

PEFT config:
{'default': LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping={'base_model_class': 'Qwen2ForCausalLM', 'parent_library': 'transformers.models.qwen2.modeling_qwen2', 'unsloth_fixed': True}, peft_version='0.19.1', base_model_name_or_path='unsloth/Qwen2.5-3B-Instruct', revision=None, inference_mode=False, r=16, target_modules={'gate_proj', 'v_proj', 'down_proj', 'q_proj', 'o_proj', 'up_proj', 'k_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, laye

In [18]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 13
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.274624
2,3.660396
3,2.787696
4,2.206186
5,2.182842
6,1.562020
7,1.947377
8,1.607489
9,1.754364
10,1.863206


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-13/tokenizer_config.json.


TrainOutput(global_step=13, training_loss=2.1637311165149393, metrics={'train_runtime': 37.119, 'train_samples_per_second': 2.694, 'train_steps_per_second': 0.35, 'total_flos': 249546029801472.0, 'train_loss': 2.1637311165149393, 'epoch': 1.0})